# Warsztaty AI: CV, CNN, YOLO

## Computer Vision

### Czym jest?

Wyobraź sobie, że pokazujesz dziecku zdjęcie kota. Dziecko od razu wie, że to kot - rozpoznaje uszy, futro, wąsy. Computer Vision to dziedzina, która próbuje nauczyć tego
samego komputery.

To gałąź sztucznej inteligencji, która pozwala maszynom:
  - widzieć obrazy i filmy (przez kamery, zdjęcia, skany),
  - rozumieć, co się na nich znajduje,
  - podejmować decyzje na podstawie tego, co "zobaczyły".

<img width="400" src="img/cat.png">

Źródło: https://www.greensboro.carolinavet.com/site/greensboro-specialty-veterinary-blog/2023/03/15/how-to-choose-cat-breed

### Gdzie spotykamy na codzień?

Face ID - rozpoznawanie twarzy

<img width="400" src="img/face_id.png">

Automiczne samochody, taksówki (np. Waymo)

<img width="400" src="img/waymo.png">

Kasy samoobsługowe

<img width="400" src="img/self_service.png">



In [ ]:
# Instalacja bibliotek
# %pip install -q numpy matplotlib pillow torch torchvision ultralytics

#### Jak komputer widzi obraz?

Dla człowieka obraz to scena. Dla komputera obraz to **macierz liczb**.

W skali szarości obraz to macierz $I \in \mathbb{R}^{H \times W}$, gdzie każdy element $i_{y,x} \in [0, 255]$ koduje jasność piksela:
- **0** - czarny,
- **255** - biały,
- wartości pośrednie - odcienie szarości.

> ℹ️ W przypadku obrazów kolorowych (RGB) mamy **trzy takie macierze ułożone na sobie** - po jednej dla kanału czerwonego, zielonego i niebieskiego. Zasada jest dokładnie ta sama.

##### Przykład: obraz 8×8 z czarnym kwadratem w środku

$$
I = \begin{bmatrix}
255 & 255 & 255 & 255 & 255 & 255 & 255 & 255 \\
255 & 255 & 255 & 255 & 255 & 255 & 255 & 255 \\
255 & 255 & 0   & 0   & 0   & 0   & 255 & 255 \\
255 & 255 & 0   & 0   & 0   & 0   & 255 & 255 \\
255 & 255 & 0   & 0   & 0   & 0   & 255 & 255 \\
255 & 255 & 0   & 0   & 0   & 0   & 255 & 255 \\
255 & 255 & 255 & 255 & 255 & 255 & 255 & 255 \\
255 & 255 & 255 & 255 & 255 & 255 & 255 & 255 \\
\end{bmatrix}
$$

I to wszystko - żadnej magii, po prostu **liczby w siatce**. Zadaniem Computer Vision jest wydobyć z tej macierzy znaczenie ("tu jest kwadrat").

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Obraz 8×8 w skali szarości - białe tło (255)
img = np.full((8, 8), 255, dtype=np.uint8)

### Ćwiczenie: narysuj czarny kwadrat

Mamy biały obraz 8×8 pikseli. Twoim zadaniem jest narysować w nim **czarny kwadrat 4×4** umieszczony **dokładnie w środku**.

In [ ]:
# TODO: ustaw indeksy tak, żeby czarny kwadrat 4×4 znalazł się w środku obrazu 8×8
#
# W NumPy: img[a:b, c:d] = 0  ustawia wartość 0 (czarny) w wierszach a..b-1 i kolumnach c..d-1
# Przykład 1: img[0:3, 0:3] = 0   →  czarny kwadrat 3×3 w lewym górnym rogu
# Przykład 2: img[1:4, 5:8] = 0   →  czarny prostokąt w prawej górnej części
#
img[___, ___] = 0

In [ ]:
# Wizualizacja: obraz + jego macierzowa reprezentacja
fig, axes = plt.subplots(1, 2, figsize=(10, 5))

axes[0].imshow(img, cmap="gray", vmin=0, vmax=255)
axes[0].set_title("Obraz 8×8 px")
axes[0].axis("off")

axes[1].imshow(img, cmap="gray", vmin=0, vmax=255)
axes[1].set_title("Ta sama macierz z wartościami pikseli")
axes[1].axis("off")
for y in range(8):
    for x in range(8):
        color = "white" if img[y, x] == 0 else "black"
        axes[1].text(
            x, y, str(img[y, x]), ha="center", va="center", color=color, fontsize=10
        )

plt.tight_layout()
plt.show()

print("Kształt macierzy:", img.shape)  # (8, 8)
print("Liczba pikseli:  ", img.size)  # 64
print("\nPełna macierz obrazu:")
print(img)

#### Skoro obraz to liczby... to można na nich robić matematykę!

To kluczowa intuicja. Większość "klasycznych" operacji graficznych (filtry w Photoshopie, efekty w Instagramie) to tak naprawdę **proste działania arytmetyczne na macierzy pikseli**.

**Przykład: inwersja (negatyw)**

$$
I' = 255 - I
$$

Bierzemy każdy piksel i odejmujemy jego wartość od 255 - czarny (0) staje się biały (255), biały (255) staje się czarny (0), a wszystko pośrodku - odwrócone.

> **Wniosek:** zaawansowane Computer Vision (CNN, YOLO, U-Net) to **te same operacje matematyczne na macierzach**, tylko dużo bardziej złożone - i sieć **sama uczy się**, jakie operacje są przydatne, zamiast je ręcznie projektować.

### Ćwiczenie: zrób negatyw zdjęcia

Wczytaj prawdziwe zdjęcie i przekształć je w **negatyw (inwersję)**. To dosłownie jedna linijka matematyki na macierzy pikseli.

In [ ]:
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt

# Wczytujemy zdjęcie jako macierz (tensor H × W × 3)
img = np.array(Image.open("img/cat.png").convert("RGB"))
print("Kształt obrazu:", img.shape, "→ H × W × kanały (RGB)")

# TODO: zapisz wzór na inwersję (negatyw)
#
# W NumPy operacje arytmetyczne działają na każdym pikselu naraz:
#   img + 50      →  rozjaśnia obraz (każdy piksel +50)
#   img - 30      →  przyciemnia obraz (każdy piksel -30)
#   stała - img   →  każdy piksel staje się "stała minus jego wartość"
#
img_inverted = ___

In [ ]:
# Teraz wyświetlmy obrazek

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].imshow(img)
axes[0].set_title("Oryginał")
axes[0].axis("off")

axes[1].imshow(img_inverted)
axes[1].set_title("Negatyw")
axes[1].axis("off")

plt.tight_layout()
plt.show()

#### Czego **nie da się** zrobić jedną prostą operacją?

Dodawanie, odejmowanie, threshold - to wszystko świetnie działa, gdy zadanie da się **opisać wzorem**:

✅ "Rozjaśnij obraz" → $I + 80$
✅ "Zrób negatyw" → $255 - I$
✅ "Wykryj ciemne piksele" → $I < 50$

Ale spróbujmy napisać wzór na:

❌ "Czy na tym zdjęciu **jest kot**?"
❌ "**Gdzie** na zdjęciu znajduje się pieszy?"
❌ "Czy ta osoba to **właściciel telefonu**?"

Nikt nie potrafi tego napisać ręcznie. "Kot" to nie jest formuła - to złożony układ kształtów, faktur, proporcji, oświetlenia. Każdy kot wygląda inaczej, a my i tak go rozpoznajemy.

> **Skoro nie umiemy napisać reguły ręcznie - pozwólmy komputerowi, żeby sam ją wymyślił, ucząc się z przykładów.**

To jest właśnie pomysł na **uczenie maszynowe** i **sieci neuronowe**.

## Typy problemów w Computer Vision

Zanim zaczniemy pisać konkretne sieci, warto wiedzieć, **na jakie pytania w ogóle staramy się odpowiadać** w CV. Te same dane wejściowe (obraz), ale **całkiem inne wyjście** - i całkiem inna architektura sieci.

Cztery klasyczne typy zadań, ułożone od najprostszego do najtrudniejszego:

### 1. Klasyfikacja (Classification)

> *"Co jest na obrazku?"*

- **Wejście:** obraz
- **Wyjście:** **jedna etykieta** dla całego obrazu
- **Założenie:** na obrazie jest jeden główny obiekt
- **Przykład sieci:** ResNet, VGG, AlexNet
- **Datasety:** MNIST (cyfry), ImageNet (1000 kategorii)

```
┌──────────────┐
│              │
│      🐱      │   →   "kot"   (pewność: 0.94)
│              │
└──────────────┘
```

### 2. Klasyfikacja + Lokalizacja (Classification + Localization)

> *"Co jest na obrazku **i gdzie dokładnie**?"*

- **Wejście:** obraz
- **Wyjście:** **jedna etykieta + jedna ramka** (bounding box)
- **Założenie:** wciąż jeden główny obiekt - ale dodatkowo chcemy wiedzieć, gdzie jest
- **Zastosowanie:** np. wyrównanie zdjęcia twarzy przed Face ID

```
┌──────────────┐
│   ┌────┐     │   →   "kot"   ramka: (x=80, y=40, w=120, h=140)
│   │ 🐱 │     │
│   └────┘     │
└──────────────┘
```

### 3. Detekcja obiektów (Object Detection)

> *"Co i gdzie - dla **wielu** obiektów jednocześnie?"*

- **Wejście:** obraz
- **Wyjście:** **lista ramek**, każda z etykietą i pewnością
- **Założenie:** na obrazie może być dowolna liczba obiektów (lub żaden)
- **Przykład sieci:** **YOLO**, Faster R-CNN, RetinaNet
- **Zastosowania:** samochody autonomiczne, kasy samoobsługowe, monitoring

```
┌──────────────┐
│ ┌──┐   ┌──┐  │   →   [("kot",  ramka_1),
│ │🐱│   │🐶│  │        ("pies", ramka_2),
│ └──┘   └──┘  │        ("piłka", ramka_3)]
│       o      │
└──────────────┘
```

### 4. Segmentacja instancji (Instance Segmentation)

> *"Co, gdzie - i **które dokładnie piksele** należą do każdego obiektu osobno?"*

- **Wejście:** obraz
- **Wyjście:** dla każdego obiektu - **maska pikselowa** (zamiast prostokątnej ramki)
- **Najtrudniejszy z czterech** - sieć musi wiedzieć, czy każdy piksel z osobna należy do jakiegoś obiektu i którego
- **Przykład sieci:** Mask R-CNN, YOLO-seg
- **Zastosowania:** medycyna (wycinanie nowotworów), efekty specjalne, autonomia

```
┌──────────────┐
│  ▓▓▓   ▒▒▒   │   →   [("kot",  maska pikseli #1: ▓▓▓),
│ ▓▓▓▓   ▒▒▒▒  │        ("pies", maska pikseli #2: ▒▒▒)]
│  ▓▓     ▒▒   │
└──────────────┘    ↑ maska na poziomie KAŻDEGO piksela
```

### Podsumowanie - jak rośnie złożoność wyjścia

| Zadanie | Wyjście | Liczba obiektów | Kształt opisu |
|---|---|---|---|
| **Klasyfikacja** | etykieta | 1 (cały obraz) | - |
| **Klas. + Lokalizacja** | etykieta + ramka | 1 | prostokąt |
| **Detekcja obiektów** | lista (etykieta + ramka) | wiele | prostokąt |
| **Segmentacja instancji** | lista (etykieta + maska) | wiele | dokładnie piksele |

<img src="img/cv_problems.png">

Źródło: https://blog.roboflow.com/object-detection-vs-image-classification-vs-keypoint-detection/

## Sieci neuronowe - w pigułce

Zanim wejdziemy w CNN, dwa zdania o tym, czym w ogóle jest sieć neuronowa.

### Neuron - najmniejszy klocek

Sztuczny neuron to **bardzo prosta funkcja matematyczna**. Bierze kilka liczb na wejściu, mnoży je przez **wagi**, sumuje i przepuszcza przez "aktywację":

$$
y = f(w_1 x_1 + w_2 x_2 + \dots + w_n x_n + b)
$$

- $x_i$ - wejścia (np. piksele),
- $w_i$ - **wagi** (to ich sieć się uczy),
- $b$ - przesunięcie (bias),
- $f$ - funkcja aktywacji (np. zwraca 0 jeśli suma jest ujemna).

### Funkcja aktywacji - czemu ona w ogóle jest?

Mnożenie, sumowanie, dodawanie biasu - to wszystko operacje **liniowe**. Funkcja aktywacji $f$ to **jedyna nieliniowa rzecz** w neuronie. I to ona daje sieci "moc".

> 💡 **Bez aktywacji** cała sieć zachowywałaby się jak **jedna linia**. Matematyka: kombinacja liniowa funkcji liniowych jest dalej liniowa. Nie nauczyłaby się rozpoznać kota, bo "kot" to nie liniowa kombinacja pikseli.

#### Najpopularniejsza: ReLU

$$
\text{ReLU}(x) = \max(0, x)
$$

Czyli:
- jeśli wejście jest **dodatnie** → przekaż dalej bez zmian,
- jeśli **ujemne** → wyzeruj.

Prosta, szybka, działa świetnie. **Tej używamy w naszej `MnistCNN`** (`F.relu(...)` w kodzie).

#### Inne, które warto znać

<img src="img/act_functions.png" width="800">

Źródło: https://medium.com/@shrutijadon/survey-on-activation-functions-for-deep-learning-9689331ba092

> 🧠 **Intuicja:** wyobraź sobie neuron jako przełącznik. Suma ważona mówi *"jak mocno jestem podekscytowany?"*. Aktywacja decyduje: *"czy w ogóle zapalić światło?"* (ReLU), *"jak jasno świecić?"* (Sigmoid).

### Sieć - wiele neuronów ułożonych w warstwy

Pojedynczy neuron jest słaby. Ale **wiele neuronów ułożonych w warstwy** potrafi modelować dowolnie skomplikowane zależności.

<img width="1000px" src="img/nn_schema.png">

Źródło: https://developers.google.com/machine-learning/crash-course/neural-networks/multi-class

### Trening - jak sieć się uczy?

1. Pokazujemy sieci **przykład** (np. zdjęcie jabłka) i **poprawną odpowiedź** ("jabłko").
2. Sieć daje jakąś odpowiedź - najpierw losową, np. jabłko.
3. Obliczamy **błąd** i delikatnie **poprawiamy wagi** w stronę lepszej odpowiedzi.
4. Powtarzamy **miliony razy** na milionach przykładów.

> **Kluczowa obserwacja:** wagi $w_i$ to też **liczby w macierzach**. Cała sieć neuronowa to po prostu **wielkie mnożenia macierzy**, jedno po drugim. Wracamy do tego samego, co przy obrazie - matematyka na macierzach, tylko więcej.

### Ćwiczenie: policz wyjście neuronu

Wzór jednego neuronu (ten sam, co na poprzednim slajdzie):

$$
y = \text{ReLU}(w_1 x_1 + w_2 x_2 + w_3 x_3 + b)
$$

Uzupełnij **jedną linijkę** kodu.

In [ ]:
import numpy as np

x = np.array([1.0, 0.5, 2.0])  # wejścia
w = np.array([0.3, -1.0, 0.5])  # wagi
b = 0.2  # bias

# TODO: oblicz y = ReLU(w·x + b)
# Podpowiedź:  w·x w NumPy  →  np.dot(w, x)
#              ReLU(x)       →  max(0, x)
y = ___

print(f"Wyjście neuronu: {y:.2f}")

## CNN - sieci konwolucyjne dla obrazów

Mamy już dwa kawałki układanki:

🧩 **Obraz** = macierz liczb, na której można robić matematyczne operacje (rozjaśnianie, threshold, filtry...).

🧩 **Sieć neuronowa** = stos warstw, które uczą się matematycznych operacji z danych.

**CNN (Convolutional Neural Network)** to po prostu **sieć neuronowa specjalnie zaprojektowana dla obrazów**. Zamiast traktować zdjęcie jako "płaski wektor liczb", wykorzystuje fakt, że to **macierz 2D z lokalną strukturą** (sąsiednie piksele są ze sobą powiązane).

### Co robi CNN - w skrócie

Zamiast jednej wielkiej operacji, CNN składa się z **wielu warstw konwolucyjnych**, gdzie każda:

1. **Stosuje filtry** do różnych fragmentów obrazu,
2. ale **filtry są uczone z danych**, a nie zaprojektowane ręcznie.

### Hierarchia

Im głębsza warstwa, tym bardziej **abstrakcyjne** cechy wykrywa:

| Warstwa | Co wykrywa |
|---|---|
| 1 (płytka) | krawędzie, kolory, proste tekstury |
| 2–3 (środkowa) | kąty, łuki, kółka, paski |
| 4–5 | fragmenty obiektów: oko, ucho, koło samochodu |
| ostatnia (głęboka) | całe obiekty: "kot", "pies", "samochód" |

<img src="img/cnn.png" width="1000">

> ℹ️ To bardzo podobne do tego, jak działa **kora wzrokowa w mózgu** - najpierw rozpoznajemy krawędzie i plamy, dopiero potem mózg składa z nich obiekty.

### Dlaczego CNN, a nie zwykła sieć?

- **Mniej wag** - filtry są małe (np. 3×3) i wielokrotnie używane na całym obrazie.
- **Niezmienniczość przestrzenna** - kot rozpoznawany jest niezależnie od tego, czy jest w lewym górnym, czy w prawym dolnym rogu.
- **Zachowuje strukturę 2D** - sąsiedztwo pikseli ma znaczenie.

W kolejnym kroku zobaczymy **co to właściwie znaczy "filtr" i "konwolucja"** - na konkretnym przykładzie.

### Filtr 3×3 w akcji - wykrywanie krawędzi

Zanim sieć nauczy się sama, zobaczmy **jak ręcznie zaprojektowany filtr 3×3** wydobywa z obrazu konkretne cechy. Klasyczny przykład: **filtry Sobela** - jeden wykrywa krawędzie **pionowe** (gwałtowna zmiana jasności w poziomie), drugi krawędzie **poziome**.

Konwolucja = przesuwamy okienko 3×3 po całym obrazie, w każdym miejscu mnożymy element po elemencie i sumujemy. Dokładnie to robi `nn.Conv2d` w CNN - tylko że tam wartości filtrów **sieć uczy się sama z danych**.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

# Wczytujemy zdjęcie w skali szarości (jedna macierz H × W)
img = np.array(Image.open("img/cat.png").convert("L"), dtype=np.float32)

# Filtr Sobela - krawędzie PIONOWE (różnica jasności: lewo vs prawo)
kernel_v = np.array(
    [
        [-1, 0, 1],
        [-2, 0, 2],
        [-1, 0, 1],
    ],
    dtype=np.float32,
)

# Filtr Sobela - krawędzie POZIOME (różnica jasności: góra vs dół)
kernel_h = np.array(
    [
        [-1, -2, -1],
        [0, 0, 0],
        [1, 2, 1],
    ],
    dtype=np.float32,
)


def conv2d(image, kernel):
    """Ręczna konwolucja - filtr 3×3 przesuwany po całym obrazie."""
    H, W = image.shape
    out = np.zeros((H - 2, W - 2), dtype=np.float32)
    for i in range(3):
        for j in range(3):
            out += kernel[i, j] * image[i : H - 2 + i, j : W - 2 + j]
    return np.abs(out)


edges_v = conv2d(img, kernel_v)
edges_h = conv2d(img, kernel_h)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(img, cmap="gray")
axes[0].set_title("Oryginał (skala szarości)")
axes[0].axis("off")

axes[1].imshow(edges_v, cmap="gray")
axes[1].set_title("Krawędzie PIONOWE (Sobel X)")
axes[1].axis("off")

axes[2].imshow(edges_h, cmap="gray")
axes[2].set_title("Krawędzie POZIOME (Sobel Y)")
axes[2].axis("off")

plt.tight_layout()
plt.show()

print("Filtr na krawędzie pionowe:")
print(kernel_v)
print("\nFiltr na krawędzie poziome:")
print(kernel_h)


## CNN w akcji - rozpoznawanie cyfr (MNIST)

Czas zobaczyć działającą sieć. Wytrenujemy małą CNN, żeby rozpoznawała ręcznie pisane cyfry **0–9**.

### Dane: MNIST

**MNIST** to klasyczny zbiór danych w uczeniu maszynowym (taki "Hello World" dla deep learning):
- **70 000 zdjęć** ręcznie pisanych cyfr,
- każde **28 × 28 px** w skali szarości,
- każde z etykietą **0–9**.

60 000 zdjęć użyjemy do **treningu**, 10 000 do **testu** (sprawdzenia, czy sieć faktycznie się nauczyła).

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import matplotlib.pyplot as plt

In [ ]:
# Pobieramy zbiór danych MNIST
transform = transforms.ToTensor()
train_data = datasets.MNIST(
    root="./data", train=True, download=True, transform=transform
)
test_data = datasets.MNIST(
    root="./data", train=False, download=True, transform=transform
)

print(f"Zbiór treningowy: {len(train_data)} obrazów")
print(f"Zbiór testowy:    {len(test_data)} obrazów")
print(f"Kształt obrazu:   {train_data[0][0].shape}  (1 kanał, 28×28 px)")

In [ ]:
# Podgląd: 10 pierwszych przykładów z etykietami
fig, axes = plt.subplots(1, 10, figsize=(15, 2))
for ax, (image, label) in zip(axes, train_data):
    ax.imshow(image.squeeze(), cmap="gray")
    ax.set_title(f"Cyfra: {label}")
    ax.axis("off")
plt.tight_layout()
plt.show()

### Architektura sieci

Sieć którą zbudujemy ma 5 warstw - kontynuacja "MnistCNN", tylko trochę większa, żeby udźwignęła 10 klas:

```
Obraz 28×28 (1 kanał)
        │
        ▼
[Konwolucja 3×3, 16 filtrów]  →  16 map 26×26   ← wykrywanie krawędzi, kształtów
        │
        ▼
[MaxPool 2×2]                 →  16 map 13×13   ← redukcja rozdzielczości
        │
        ▼
[Konwolucja 3×3, 32 filtry]   →  32 mapy 11×11  ← wykrywanie bardziej złożonych cech
        │
        ▼
[MaxPool 2×2]                 →  32 mapy 5×5
        │
        ▼
[Spłaszczenie]                →  wektor 800 liczb
        │
        ▼
[Warstwa klasyfikująca]       →  10 liczb (po jednej na każdą cyfrę 0–9)
```

#### Po co jaka warstwa?

- **`Conv2d`** - wykrywa wzorce. Każdy filtr 3×3 szuka konkretnej cechy (krawędź, kształt, fragment obiektu). Im głębsza warstwa, tym bardziej złożone wzorce.
- **`MaxPool2d`** - zmniejsza obraz 2× w każdym wymiarze. Mniej obliczeń + odporność na drobne przesunięcia.
- **Naprzemiennie Conv → Pool** - sieć stopniowo "oddala kamerę": od pikseli (784 liczby) do abstrakcyjnych cech (800 liczb, ale każda znaczy coś bardziej złożonego).
- **`Linear`** - końcowy klasyfikator. Z 800 cech robi 10 liczb (po jednej na cyfrę). Najwyższa = predykcja.

Trenujemy przez **1 epokę** (1 przejście przez wszystkie 60k obrazów).

### Ćwiczenie: dokończ definicję sieci

W kodzie poniżej masz definicję klasy `MnistCNN`. Jedno miejsce wymaga uzupełnienia - **liczba klas wyjściowych** w warstwie `fc`.

In [ ]:
class MnistCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 16, kernel_size=3)
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3)
        self.pool = nn.MaxPool2d(kernel_size=2)
        # TODO: ile klas rozpoznajemy? (cyfry od 0 do 9)
        self.fc = nn.Linear(32 * 5 * 5, ___)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = x.flatten(start_dim=1)
        return self.fc(x)


model = MnistCNN()
print(model)
print(f"\nLiczba uczonych parametrów: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
# Optymalizator (algorytm dobierający wagi) i funkcja straty (mierzy, jak bardzo się mylimy)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
loss_fn = nn.CrossEntropyLoss()

# Loader treningowy: dzieli 60 000 obrazów na batche po 128
train_loader = DataLoader(train_data, batch_size=128, shuffle=True)
print(f"Liczba batchy w jednej epoce: {len(train_loader)}")

In [ ]:
# Trening - 1 epoka (jedno przejście przez wszystkie 60 000 obrazów)
model.train()
for i, (images, labels) in enumerate(train_loader):
    optimizer.zero_grad()  # zerujemy gradienty z poprzedniego kroku
    predictions = model(images)  # forward pass
    loss = loss_fn(predictions, labels)
    loss.backward()  # liczymy gradienty (jak poprawić wagi)
    optimizer.step()  # aktualizujemy wagi

    if i % 100 == 0:
        print(f"  batch {i:4d}/{len(train_loader)}   loss = {loss.item():.4f}")

print("\nTrening zakończony.")

In [ ]:
# Ewaluacja - sprawdzamy skuteczność na 10 000 obrazów, których sieć NIGDY nie widziała
test_loader = DataLoader(test_data, batch_size=1000)
model.eval()

correct, total = 0, 0
with torch.no_grad():
    for images, labels in test_loader:
        output = model(images)
        predictions = output.argmax(dim=1)
        correct += (predictions == labels).sum().item()
        total += labels.size(0)

accuracy = correct / total * 100
print(f"Skuteczność na zbiorze testowym: {accuracy:.2f}%  ({correct}/{total})")

In [ ]:
# Podgląd predykcji na 16 losowych przykładach z testu
import random

model.eval()
fig, axes = plt.subplots(2, 8, figsize=(16, 4))
for ax in axes.flatten():
    idx = random.randint(0, len(test_data) - 1)
    image, true_label = test_data[idx]
    with torch.no_grad():
        prediction = model(image.unsqueeze(0)).argmax(dim=1).item()

    is_correct = prediction == true_label
    color = "green" if is_correct else "red"
    ax.imshow(image.squeeze(), cmap="gray")
    ax.set_title(f"praw: {true_label}\npred: {prediction}", color=color, fontsize=10)
    ax.axis("off")

plt.tight_layout()
plt.show()

## YOLO - od klasyfikacji do detekcji

<img src="img/yolo.gif">

Źródło: https://medium.com/data-science/enhanced-object-detection-how-to-effectively-implement-yolov8-afd1bf6132ae


Nasza `CNNcyfr` odpowiadała na pytanie:

> *"**Co** jest na obrazku?"* → **jedna** klasa dla całego obrazu (np. "cyfra 7")

To **klasyfikacja**. Ale w wielu zadaniach to za mało:

🚗 Samochód autonomiczny: musi wiedzieć, **gdzie dokładnie** są pieszy, znak, inny samochód.
📦 Magazyn Amazonu: musi policzyć **każdą** paczkę osobno.
🏥 Diagnostyka: musi zaznaczyć **konkretne miejsce** zmiany na zdjęciu RTG.

To jest **detekcja obiektów** - odpowiedź na pytanie:

> *"**Co** i **gdzie** jest na obrazku?"* → **wiele** ramek z klasami

### Czym jest YOLO?

**YOLO** = **Y**ou **O**nly **L**ook **O**nce. To rodzina sieci CNN zaprojektowanych specjalnie do detekcji obiektów. Kluczowa cecha:

> Cały obraz "przechodzi" przez sieć **jednym forward passem** - w jednym przebiegu YOLO zwraca **wszystkie** ramki, klasy i pewności jednocześnie.

Dzięki temu działa **bardzo szybko** - nawet kilkadziesiąt klatek na sekundę. Inne podejścia (np. R-CNN) robiły to wieloetapowo i były dużo wolniejsze.

<img src="img/yolo.png" width="1000">

### Pod spodem to nadal CNN

YOLO to **ta sama idea co `CNNcyfr`** - stos warstw konwolucyjnych - tylko z innym **wyjściem**. Zamiast 10 liczb (po jednej na klasę), zwraca dla każdej "komórki" w obrazie:
- czy jest tam obiekt,
- jaka jest klasa,
- współrzędne ramki `(x, y, szerokość, wysokość)`.

### W praktyce

Wykorzystamy **pre-trained model** (yolov8n) wytrenowany na **COCO** - zbiorze z 80 klasami obiektów codziennych (człowiek, samochód, pies, butelka, ...). To tzw. **transfer learning** - używamy gotowych "oczu" do swoich zadań.

### Co było przed YOLO?

Wcześniejsze podejścia do detekcji (np. **R-CNN** i jego następcy) działały **dwuetapowo**:

1. **Znajdź potencjalne regiony** - algorytm proponuje setki, czasem tysiące prostokątów, w których "coś może być".
2. **Sklasyfikuj każdy region osobno** - każdy kandydat puszczany przez CNN.

Czyli dla jednego obrazu robiło się **setki/tysiące przebiegów sieci**. To było **powolne** - detekcja na żywo (wideo, kamera) była technicznie niewykonalna.

#### Co YOLO zrobił inaczej

YOLO patrzy na obraz **JEDEN raz**, dzieli go na siatkę, a dla każdej komórki sieć **jednocześnie** przewiduje: *czy jest tam obiekt, jaki, gdzie ma ramkę*. Wszystko w jednym forward passie sieci. Stąd nazwa: **You Only Look Once**.

Efekt: detekcja w **kilkadziesiąt klatek na sekundę** - czyli real-time. Dzięki temu możliwe stały się: samochody autonomiczne, monitoring na żywo, filtry AR.

#### Z czym YOLO radzi sobie **gorzej**

YOLO to nie "lepszy we wszystkim" - to **kompromis szybkości i precyzji**. Wolniejsze 2-etapowe modele bywają lepsze w kilku sytuacjach:

- 🔍 **Małe obiekty** - np. dalekie znaki drogowe, drobne zmiany na zdjęciu medycznym
- 👥 **Obiekty stłoczone razem** - gdy kilka obiektów wpada do tej samej "komórki siatki" (np. tłum twarzy), YOLO może niektóre pominąć
- 📏 **Nietypowe proporcje** - bardzo wąskie/wysokie albo bardzo szerokie kształty
- 🎯 **Surowa precyzja na benchmarkach** - Faster R-CNN / Mask R-CNN bywają minimalnie dokładniejsze (kosztem 10–100× większego czasu inferencji)

W praktyce dla większości zastosowań **YOLO wystarczy z naddatkiem**, a 100× większa prędkość przeważa nad marginalną stratą precyzji.

In [ ]:
from ultralytics import YOLO

# Wczytujemy pre-trained model YOLOv8 nano (najmniejszy, ~6 MB)
# Przy pierwszym uruchomieniu zostanie pobrany automatycznie
yolo_model = YOLO("yolov8n.pt")

# Lista klas, które model umie rozpoznawać (80 klas ze zbioru COCO)
print(f"Liczba klas, które YOLO zna: {len(yolo_model.names)}")
print(f"Pierwsze 10 klas: {list(yolo_model.names.values())[:10]}")

In [ ]:
# Uruchamiamy detekcję na naszym zdjęciu
results = yolo_model("img/cat.png")

# Wyświetlamy obraz z narysowanymi ramkami
import matplotlib.pyplot as plt

annotated_image = results[0].plot()  # zwraca numpy array (BGR)
plt.figure(figsize=(12, 8))
plt.imshow(annotated_image[..., ::-1])  # BGR → RGB
plt.axis("off")
plt.show()

### Co YOLO faktycznie zwraca?

Obrazek z ramkami wygląda imponująco, ale "pod spodem" YOLO zwraca **zwykłą strukturę danych** - listę wykrytych obiektów. Dla każdego:

- **Ramka (bounding box)** - 4 liczby: `(x1, y1, x2, y2)` lub `(x_środek, y_środek, szerokość, wysokość)`
- **Klasa** - indeks klasy (np. `0` = "person", `2` = "car") + nazwa
- **Pewność (confidence)** - od `0.0` do `1.0`, jak bardzo model jest pewien

In [ ]:
# Surowe wyniki detekcji
detections = results[0].boxes

print(f"Wykryto {len(detections)} obiektów\n")

for i, box in enumerate(detections):
    class_idx = int(box.cls[0])  # id klasy
    class_name = yolo_model.names[class_idx]  # nazwa klasy
    confidence = float(box.conf[0])  # liczba od 0.0 do 1.0
    x1, y1, x2, y2 = box.xyxy[0].tolist()

    print(
        f"  {i + 1}. {class_name:15s}  pewność: {confidence:.2%}  "
        f"ramka: ({x1:.0f}, {y1:.0f}) → ({x2:.0f}, {y2:.0f})"
    )

### Ćwiczenie: policz koty na zdjęciu

To **najtrudniejsze** ćwiczenie. Mamy zdjęcie z wieloma kotami (`many_cats.png`). Twoim zadaniem jest:

1. Uruchomić YOLO na zdjęciu i obejrzeć, co wykrył.
2. **Samodzielnie napisać pętlę**, która policzy, ile **kotów** wykryło YOLO **z pewnością powyżej 50%**.


In [ ]:
# Detekcja na zdjęciu z wieloma kotami
cats_results = yolo_model("img/many_cats.png")

# Pokaż obraz z narysowanymi ramkami
annotated_image = cats_results[0].plot()
plt.figure(figsize=(12, 8))
plt.imshow(annotated_image[..., ::-1])
plt.axis("off")
plt.show()

# Wszystkie detekcje wykryte przez YOLO na tym zdjęciu
detections = cats_results[0].boxes

# TODO: napisz pętlę, która policzy koty z pewnością > 50%

cat_count = 0

# TODO

print(f"YOLO wykryło {cat_count} kotów z pewnością powyżej 50%")
